# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset on second primary colorectal cancer using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described via a Croissant schema at the following URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
List all available record sets, their fields, and columns by their `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets.values())
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description if hasattr(rs, 'description') else '-'}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}) | type: {getattr(field, 'data_type', '-')}")
            if hasattr(field, 'columns') and field.columns:
                print("      Columns:")
                for col in field.columns:
                    print(f"        * {col.name} (@id: {col.id}) type: {getattr(col, 'data_type', '-')}" )
    print()

## 3. Data Extraction
Load the data for the primary record set into a pandas DataFrame using their `@id`.

**Note:** We will extract all available record sets. For this dataset, there is typically one main record set representing the clinical tabular data.

In [ ]:
# Get the @ids of all available record sets
record_set_ids = [rs.id for rs in record_sets]
print("Record sets IDs:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Preview the main DataFrame (usually only one for clinical tabular dataset)
main_record_set_id = record_set_ids[0]  # or change if multiple present
print(f"\nColumns for main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group the data using field `@id`s.

In [ ]:
# Set up field/column @ids to use (replace as appropriate)

# Use first numeric field found for demonstration (e.g., age at diagnosis)
df = dataframes[main_record_set_id]
numeric_field_id = None
for col in df.columns:
    # Heuristic: Look for columns with numeric types (excluding IDs)
    if pd.api.types.is_numeric_dtype(df[col]) and not col.endswith('_id'):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: Try a known column name
    numeric_field_id = df.columns[0]
print(f"Selected numeric field @id for EDA: {numeric_field_id}")

# Filter: All values where the field is above its mean
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered ({len(filtered_df)}) records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Demo: select a string/categorical/grouping field (e.g. cancer_site or MSI status)
group_field_id = None
for col in df.columns:
    if df[col].dtype == 'O' and not col.endswith('_id'):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a key numeric field and group statistics.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of the main numeric field
plt.figure(figsize=(7,4))
df[numeric_field_id].hist(bins=15, alpha=0.8)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field was found, show grouped means as bar chart
if group_field_id:
    plt.figure(figsize=(8,4))
    grouped_df.plot(x=group_field_id, y=numeric_field_id, kind='bar', legend=False, color='cornflowerblue', ax=plt.gca())
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- Loaded and parsed the FAIR^2 dataset using `mlcroissant`.
- Inspected record sets, fields, and columns by their `@id`.
- Performed data extraction into `pandas.DataFrame` using record set and field `@id`s.
- Demonstrated basic exploratory analysis: filtering on a numeric field, normalization, and grouping.
- Visualized numeric data distributions and group statistics.

This workflow provides a reproducible, schema-driven approach for processing research datasets with machine-readability and traceability to Croissant `@id`s for each entity.